In [ ]:
# ================================================================
# COMFYQUEUE WORKER — Célula 1: Configuração
# ================================================================
# Ajuste as variáveis abaixo e execute esta célula primeiro.
# ================================================================

LARAVEL_API_URL = "https://seu-portal.com"  # URL base do servidor Laravel (sem barra final)
LARAVEL_API_KEY = ""                         # Valor de COMFYQUEUE_API_KEY no .env do servidor

COMFYUI_DIR  = "/content/ComfyUI"            # Diretório de instalação do ComfyUI
COMFYUI_PORT = 8188                           # Porta do ComfyUI

# Headers enviados em todas as chamadas à API do Laravel
HEADERS = {"X-Api-Key": LARAVEL_API_KEY, "Content-Type": "application/json"}

print("✓ Configuração carregada")
print(f"  Servidor : {LARAVEL_API_URL}")
print(f"  ComfyUI  : {COMFYUI_DIR}  (porta {COMFYUI_PORT})")

## Célula 2 — Instalar ComfyUI

Clona (ou atualiza) o repositório e instala as dependências Python.

In [ ]:
import os, subprocess, sys

def run(cmd, **kwargs):
    """Executa um comando shell e lança exceção se falhar."""
    subprocess.run(cmd, shell=True, check=True, **kwargs)

# Clonar ou atualizar ComfyUI
if not os.path.exists(COMFYUI_DIR):
    print("Clonando ComfyUI...")
    run(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFYUI_DIR}")
else:
    print("Atualizando ComfyUI...")
    run("git pull", cwd=COMFYUI_DIR)

# Instalar dependências Python
print("Instalando dependências...")
run(f"{sys.executable} -m pip install -r {COMFYUI_DIR}/requirements.txt -q")

print("\n✓ ComfyUI instalado e pronto")

## Célula 3 — Baixar modelos

Busca os modelos necessários dos jobs **pendentes** no Laravel e faz download apenas dos que ainda não existem no ComfyUI.

In [ ]:
import requests
from pathlib import Path

def download_models():
    resp = requests.get(f"{LARAVEL_API_URL}/api/comfy-queue/models", headers=HEADERS, timeout=30)
    resp.raise_for_status()

    models = resp.json()
    if not models:
        print("Nenhum modelo necessário para os jobs pendentes.")
        return

    print(f"{len(models)} modelo(s) necessário(s):\n")

    for model in models:
        name = model.get("name", "")
        dest = model.get("dest", "models/checkpoints")
        url  = model.get("url", "")

        if not name or not url:
            print(f"  ⚠ Entrada inválida (sem nome ou URL): {model}")
            continue

        dest_dir  = Path(COMFYUI_DIR) / dest
        dest_file = dest_dir / name
        dest_dir.mkdir(parents=True, exist_ok=True)

        if dest_file.exists():
            print(f"  ✓ {name} — já existe, pulando")
            continue

        print(f"  ↓ Baixando {name}...")
        with requests.get(url, stream=True, timeout=3600) as r:
            r.raise_for_status()
            total      = int(r.headers.get("content-length", 0))
            downloaded = 0
            with open(dest_file, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):  # chunks de 1 MB
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total:
                        print(f"    {downloaded / total * 100:.1f}%", end="\r")
        print(f"  ✓ {name} — concluído")

    print("\n✓ Todos os modelos prontos")

download_models()

## Célula 4 — Executar jobs

Inicia o ComfyUI, busca os jobs pendentes um a um, envia cada workflow ao ComfyUI e reporta o resultado (sucesso ou erro) de volta ao Laravel.

In [ ]:
import requests, subprocess, sys, time

COMFYUI_BASE = f"http://127.0.0.1:{COMFYUI_PORT}"


# ── Utilitários ───────────────────────────────────────────────

def start_comfyui():
    """Inicia o processo do ComfyUI e aguarda ficar disponível (até 3 min)."""
    print("Iniciando ComfyUI...")
    proc = subprocess.Popen(
        [sys.executable, "main.py", "--port", str(COMFYUI_PORT), "--listen", "--disable-auto-launch"],
        cwd=COMFYUI_DIR,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for attempt in range(60):
        time.sleep(3)
        try:
            r = requests.get(f"{COMFYUI_BASE}/system_stats", timeout=3)
            if r.status_code == 200:
                print(f"✓ ComfyUI disponível (tentativa {attempt + 1})")
                return proc
        except requests.exceptions.ConnectionError:
            pass
    proc.terminate()
    raise RuntimeError("ComfyUI não respondeu em 3 minutos")


def submit_workflow(workflow: dict) -> str:
    """Envia o workflow ao ComfyUI e retorna o prompt_id."""
    r = requests.post(f"{COMFYUI_BASE}/prompt", json={"prompt": workflow}, timeout=30)
    r.raise_for_status()
    return r.json()["prompt_id"]


def wait_for_result(prompt_id: str, timeout_sec: int = 600) -> dict:
    """Aguarda a conclusão do prompt e retorna o histórico."""
    deadline = time.time() + timeout_sec
    while time.time() < deadline:
        time.sleep(5)
        r = requests.get(f"{COMFYUI_BASE}/history/{prompt_id}", timeout=10)
        if r.status_code == 200:
            history = r.json()
            if prompt_id in history:
                return history[prompt_id]
    raise TimeoutError(f"Job {prompt_id} não concluído em {timeout_sec}s")


def extract_output_files(history: dict) -> list:
    """Extrai a lista de arquivos gerados do histórico do ComfyUI."""
    files = []
    for node_output in history.get("outputs", {}).values():
        for img in node_output.get("images", []):
            files.append({
                "filename":  img["filename"],
                "subfolder": img.get("subfolder", ""),
                "type":      img.get("type", "output"),
            })
    return files


# ── Loop principal ────────────────────────────────────────────

comfyui_proc = start_comfyui()
processed = 0
failed    = 0

try:
    while True:
        # Busca e reivindica o próximo job pendente
        resp = requests.get(f"{LARAVEL_API_URL}/api/comfy-queue/next", headers=HEADERS, timeout=30)

        if resp.status_code == 204:
            print("\n✓ Nenhum job pendente. Worker finalizado.")
            break

        resp.raise_for_status()
        job    = resp.json()
        job_id = job["id"]
        print(f"\n[Job #{job_id}] tipo: {job['type']}")

        try:
            prompt_id    = submit_workflow(job["params"])
            print(f"  → prompt_id: {prompt_id}")

            result       = wait_for_result(prompt_id)
            output_files = extract_output_files(result)

            requests.post(
                f"{LARAVEL_API_URL}/api/comfy-queue/job/{job_id}/done",
                headers=HEADERS,
                json={"output_files": output_files, "prompt_id": prompt_id},
                timeout=30,
            ).raise_for_status()

            processed += 1
            print(f"  ✓ Concluído — {len(output_files)} arquivo(s) gerado(s)")

        except Exception as exc:
            requests.post(
                f"{LARAVEL_API_URL}/api/comfy-queue/job/{job_id}/fail",
                headers=HEADERS,
                json={"error": str(exc)},
                timeout=30,
            )
            failed += 1
            print(f"  ✗ Falhou: {exc}")

finally:
    comfyui_proc.terminate()
    print(f"\nResumo: {processed} concluído(s), {failed} falhou(ram)")